# ITC6110 — NLP Group Project
## Notebook 3: Deep Learning Classification + RAG Pipeline + Streamlit UI

**Steps covered:** 4.2 Task 1 DL (LSTM + DistilBERT), Task 2 (RAG), Task 3 (Streamlit + HuggingFace Spaces)

**Inputs:** `data/processed/bbc_news_processed.csv`, `data/processed/best_ml_pipeline.pkl`

---
## 0. Imports & Device Setup

In [ ]:
import os, ast, pickle, warnings, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import Counter

# PyTorch (MPS)
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau

# HuggingFace
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                           Trainer, TrainingArguments, DataCollatorWithPadding)
from datasets import Dataset as HFDataset
import evaluate

# Sentence Transformers + FAISS
from sentence_transformers import SentenceTransformer
import faiss

# Sklearn utilities
from sklearn.metrics import (classification_report, confusion_matrix,
                              ConfusionMatrixDisplay, accuracy_score, f1_score)
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 120

SEED   = 42
FIGS   = Path('../outputs/figures')
MODELS = Path('../models')
DATA   = Path('../data/processed')

torch.manual_seed(SEED)
np.random.seed(SEED)

# Device: MPS > CUDA > CPU
if torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
elif torch.cuda.is_available():
    DEVICE = torch.device('cuda')
else:
    DEVICE = torch.device('cpu')

print(f"Active device: {DEVICE}")
print(f"PyTorch: {torch.__version__}")

In [ ]:
# Load processed dataset
df = pd.read_csv(DATA / 'bbc_news_processed.csv')
df['tokens'] = df['tokens'].apply(ast.literal_eval)

le = LabelEncoder()
df['label'] = le.fit_transform(df['category'])

train_df = df[df['split'] == 'train'].reset_index(drop=True)
test_df  = df[df['split'] == 'test'].reset_index(drop=True)

print(f"Train: {len(train_df)}  |  Test: {len(test_df)}")
print(f"Classes: {le.classes_}")

---
## 1. LSTM Text Classifier (Step 4.2 Task 1 — DL)

A bidirectional 2-layer LSTM trained end-to-end on the pre-processed token
sequences. We use the **lemmatised, stopword-removed** tokens from Notebook 1
and build a custom vocabulary from the training set only.

In [ ]:
# ── Build vocabulary from training tokens only (prevents leakage) ────────────
VOCAB_SIZE = 15_000
MAX_LEN    = 200

counter = Counter(tok for tokens in train_df['tokens'] for tok in tokens)
vocab = {'<PAD>': 0, '<UNK>': 1}
for word, _ in counter.most_common(VOCAB_SIZE - 2):
    vocab[word] = len(vocab)

print(f"Vocabulary size: {len(vocab):,}  (capped at {VOCAB_SIZE})")
print(f"Max sequence length: {MAX_LEN} tokens")

In [ ]:
# ── Encode tokens → padded integer sequences ──────────────────────────────────
def encode(tokens, vocab, max_len):
    ids = [vocab.get(t, 1) for t in tokens[:max_len]]
    ids += [0] * (max_len - len(ids))        # pad with <PAD> (0)
    return ids

train_enc = [encode(t, vocab, MAX_LEN) for t in train_df['tokens']]
test_enc  = [encode(t, vocab, MAX_LEN) for t in test_df['tokens']]

# ── PyTorch Dataset ───────────────────────────────────────────────────────────
class TextDataset(Dataset):
    def __init__(self, encodings, labels):
        self.X = torch.tensor(encodings, dtype=torch.long)
        self.y = torch.tensor(labels,    dtype=torch.long)
    def __len__(self):         return len(self.X)
    def __getitem__(self, idx): return self.X[idx], self.y[idx]

BATCH = 64
train_ds = TextDataset(train_enc, train_df['label'].values)
test_ds  = TextDataset(test_enc,  test_df['label'].values)

train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH)

print(f"Train batches: {len(train_loader)}  |  Test batches: {len(test_loader)}")

In [ ]:
# ── Bidirectional LSTM model ──────────────────────────────────────────────────
class BiLSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, n_classes,
                 n_layers=2, dropout=0.3):
        super().__init__()
        self.embed   = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm    = nn.LSTM(embed_dim, hidden_dim, num_layers=n_layers,
                               batch_first=True, bidirectional=True,
                               dropout=dropout)
        self.dropout = nn.Dropout(dropout)
        self.fc      = nn.Linear(hidden_dim * 2, n_classes)  # *2 for bidirectional

    def forward(self, x):
        x = self.dropout(self.embed(x))
        _, (h, _) = self.lstm(x)
        # Concatenate last forward and last backward hidden state
        h = torch.cat([h[-2], h[-1]], dim=1)
        return self.fc(self.dropout(h))

N_CLASSES = len(le.classes_)
model = BiLSTMClassifier(vocab_size=len(vocab), embed_dim=128,
                          hidden_dim=256, n_classes=N_CLASSES).to(DEVICE)
print(model)
total = sum(p.numel() for p in model.parameters())
print(f"\nTotal parameters: {total:,}")

In [ ]:
# ── Training loop ─────────────────────────────────────────────────────────────
EPOCHS = 8
optimiser  = Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler  = ReduceLROnPlateau(optimiser, patience=2, factor=0.5)
criterion  = nn.CrossEntropyLoss()

def evaluate_model(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(DEVICE), y.to(DEVICE)
            preds = model(X).argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y.cpu().numpy())
    return accuracy_score(all_labels, all_preds), all_preds, all_labels

history = {'train_loss': [], 'val_acc': []}
print(f"Training on {DEVICE}\n{'─'*45}")

for epoch in range(1, EPOCHS + 1):
    model.train()
    epoch_loss = 0
    for X, y in train_loader:
        X, y = X.to(DEVICE), y.to(DEVICE)
        optimiser.zero_grad()
        loss = criterion(model(X), y)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimiser.step()
        epoch_loss += loss.item()

    val_acc, _, _ = evaluate_model(model, test_loader)
    avg_loss = epoch_loss / len(train_loader)
    scheduler.step(avg_loss)
    history['train_loss'].append(avg_loss)
    history['val_acc'].append(val_acc)
    print(f"Epoch {epoch:2d}/{EPOCHS}  loss={avg_loss:.4f}  val_acc={val_acc:.4f}")

torch.save(model.state_dict(), MODELS / 'lstm_classifier.pt')
print("\nModel saved: models/lstm_classifier.pt")

In [ ]:
# Training curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history['train_loss'], 'o-', color='steelblue')
axes[0].set_title('Training Loss — BiLSTM')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Cross-Entropy Loss')
axes[0].grid(alpha=0.3)

axes[1].plot(history['val_acc'], 'o-', color='#2ecc71')
axes[1].set_title('Validation Accuracy — BiLSTM')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
axes[1].set_ylim(0, 1); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(FIGS / '14_lstm_training_curves.png', bbox_inches='tight')
plt.show()

In [ ]:
# Final LSTM evaluation
lstm_acc, lstm_preds, lstm_true = evaluate_model(model, test_loader)
lstm_f1 = f1_score(lstm_true, lstm_preds, average='macro')

print(f"LSTM Test Accuracy: {lstm_acc:.4f}  |  Macro F1: {lstm_f1:.4f}")
print("\n" + classification_report(lstm_true, lstm_preds, target_names=le.classes_))

fig, ax = plt.subplots(figsize=(7, 5))
ConfusionMatrixDisplay(confusion_matrix(lstm_true, lstm_preds),
                       display_labels=le.classes_).plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title(f'BiLSTM Confusion Matrix  (Acc={lstm_acc:.3f})')
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.savefig(FIGS / '15_lstm_confusion_matrix.png', bbox_inches='tight')
plt.show()

---
## 2. DistilBERT Fine-Tuning (Step 4.2 Task 1 — DL)

**DistilBERT** is a distilled version of BERT — 40% smaller, 60% faster,
retaining 97% of BERT's performance. We fine-tune it for 5-class classification.

**Important:** We use the **original** (unprocessed) article text here, not the
lemmatised version. BERT has its own subword tokenizer and performs best on
natural text with punctuation and capitalisation intact.

In [ ]:
BERT_MODEL = 'distilbert-base-uncased'
MAX_BERT   = 256       # truncate to 256 subword tokens (articles avg ~390 words)
N_CLASSES  = len(le.classes_)

tokenizer = AutoTokenizer.from_pretrained(BERT_MODEL)
print(f"Tokenizer: {BERT_MODEL}")
print(f"Vocab size: {tokenizer.vocab_size:,}")

# Build HuggingFace datasets from original text
def make_hf_dataset(split_df):
    return HFDataset.from_dict({
        'text':  split_df['text'].tolist(),
        'label': split_df['label'].tolist(),
    })

def tokenize_fn(batch):
    return tokenizer(batch['text'], truncation=True, max_length=MAX_BERT)

hf_train = make_hf_dataset(train_df).map(tokenize_fn, batched=True)
hf_test  = make_hf_dataset(test_df).map(tokenize_fn, batched=True)

print(f"\nHF Train: {len(hf_train)}  |  HF Test: {len(hf_test)}")
print(f"Features: {hf_train.features}")

In [ ]:
# Load model with a classification head
bert_model = AutoModelForSequenceClassification.from_pretrained(
    BERT_MODEL,
    num_labels=N_CLASSES,
    id2label={i: c for i, c in enumerate(le.classes_)},
    label2id={c: i for i, c in enumerate(le.classes_)},
)

trainable = sum(p.numel() for p in bert_model.parameters() if p.requires_grad)
print(f"DistilBERT parameters: {sum(p.numel() for p in bert_model.parameters()):,}")
print(f"Trainable: {trainable:,}")

In [ ]:
# Accuracy metric for Trainer
accuracy_metric = evaluate.load('accuracy')
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return accuracy_metric.compute(predictions=preds, references=labels)

# TrainingArguments — MPS auto-detected in transformers 5.x
training_args = TrainingArguments(
    output_dir=str(MODELS / 'distilbert'),
    num_train_epochs=4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='accuracy',
    logging_steps=20,
    report_to='none',
    seed=SEED,
)

print(f"Training on device: {training_args.device}")

trainer = Trainer(
    model=bert_model,
    args=training_args,
    train_dataset=hf_train,
    eval_dataset=hf_test,
    tokenizer=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
)

print("Starting DistilBERT fine-tuning...")
trainer.train()

In [ ]:
# Evaluate DistilBERT
bert_results = trainer.evaluate()
print("\nDistilBERT evaluation:")
for k, v in bert_results.items():
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

# Get predictions for confusion matrix + F1
bert_preds_out = trainer.predict(hf_test)
bert_preds = np.argmax(bert_preds_out.predictions, axis=1)
bert_true  = bert_preds_out.label_ids

bert_acc = accuracy_score(bert_true, bert_preds)
bert_f1  = f1_score(bert_true, bert_preds, average='macro')

print(f"\nDistilBERT Test Accuracy: {bert_acc:.4f}  |  Macro F1: {bert_f1:.4f}")
print("\n" + classification_report(bert_true, bert_preds, target_names=le.classes_))

fig, ax = plt.subplots(figsize=(7, 5))
ConfusionMatrixDisplay(confusion_matrix(bert_true, bert_preds),
                       display_labels=le.classes_).plot(ax=ax, colorbar=False, cmap='Greens')
ax.set_title(f'DistilBERT Confusion Matrix  (Acc={bert_acc:.3f})')
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.savefig(FIGS / '16_distilbert_confusion_matrix.png', bbox_inches='tight')
plt.show()

---
## 3. Full Model Comparison

In [ ]:
# Load ML results from Notebook 2
ml_summary = pd.read_csv(DATA / 'ml_results_summary.csv', index_col=0)

comparison = pd.DataFrame({
    'Model':    ml_summary.index.tolist() + ['BiLSTM', 'DistilBERT (fine-tuned)'],
    'Type':     ['Traditional ML'] * len(ml_summary) + ['Deep Learning', 'Deep Learning'],
    'Accuracy': ml_summary['Accuracy'].tolist() + [lstm_acc, bert_acc],
    'Macro F1': ml_summary['Macro F1'].tolist() + [lstm_f1, bert_f1],
}).sort_values('Macro F1', ascending=False).reset_index(drop=True)

print("=== Full Model Comparison ===")
print(comparison.to_string(index=False))
comparison

In [ ]:
# Bar chart comparison
fig, ax = plt.subplots(figsize=(11, 5))
x = np.arange(len(comparison))
w = 0.35
colors = ['#3498db' if t == 'Traditional ML' else '#e74c3c'
          for t in comparison['Type']]

bars1 = ax.bar(x - w/2, comparison['Accuracy'], w, label='Accuracy',
               color=colors, alpha=0.85)
bars2 = ax.bar(x + w/2, comparison['Macro F1'], w, label='Macro F1',
               color=colors, alpha=0.5)

ax.set_xticks(x)
ax.set_xticklabels(comparison['Model'], rotation=20, ha='right')
ax.set_ylim(0, 1.05)
ax.set_title('Model Comparison — Accuracy & Macro F1', fontsize=13)
ax.set_ylabel('Score')
ax.legend()
ax.grid(axis='y', alpha=0.3)

# Value labels
for bar in list(bars1) + list(bars2):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{bar.get_height():.3f}', ha='center', fontsize=7.5)

# Colour legend for ML vs DL
from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(color='#3498db', label='Traditional ML'),
    Patch(color='#e74c3c', label='Deep Learning'),
    Patch(color='gray', alpha=0.85, label='Accuracy'),
    Patch(color='gray', alpha=0.5,  label='Macro F1'),
])
plt.tight_layout()
plt.savefig(FIGS / '17_model_comparison.png', bbox_inches='tight')
plt.show()

---
## 4. Retrieval-Augmented Generation (RAG) — Step 4.2 Task 2

**Pipeline:**
1. **Retriever:** `sentence-transformers/all-MiniLM-L6-v2` encodes every article
   into a 384-dim dense vector. FAISS stores these for fast nearest-neighbour search.
2. **Prompt engineering:** The top-K retrieved articles are formatted into a
   context block and injected into a structured prompt.
3. **Generator:** `google/flan-t5-base` reads the prompt and produces the answer.

We build the RAG over **all 2,225 BBC articles** so queries about any topic are
supported, not only sport.

In [ ]:
# Load sentence encoder — MPS supported in sentence-transformers 5.x
ENCODER_MODEL = 'sentence-transformers/all-MiniLM-L6-v2'
encoder = SentenceTransformer(ENCODER_MODEL, device=str(DEVICE))
print(f"Encoder: {ENCODER_MODEL} on {DEVICE}")
print(f"Embedding dimension: {encoder.get_sentence_embedding_dimension()}")

In [ ]:
# Encode all articles (use original text for richer retrieval)
print(f"Encoding {len(df)} articles...")
article_texts = df['text'].tolist()
embeddings = encoder.encode(article_texts, batch_size=64,
                             show_progress_bar=True, normalize_embeddings=True)
print(f"Embedding matrix: {embeddings.shape}  dtype={embeddings.dtype}")

In [ ]:
# Build FAISS index (IndexFlatIP — inner product on L2-normalised vectors = cosine sim)
DIM   = embeddings.shape[1]
index = faiss.IndexFlatIP(DIM)
index.add(embeddings.astype(np.float32))

print(f"FAISS index: {index.ntotal} vectors, dim={DIM}")

# Save index and article metadata
faiss.write_index(index, str(DATA / 'faiss_index.index'))
df[['text', 'category']].to_csv(DATA / 'articles_for_rag.csv', index=False)
print("Saved: data/processed/faiss_index.index")
print("Saved: data/processed/articles_for_rag.csv")

In [ ]:
# ── Retriever ─────────────────────────────────────────────────────────────────
def retrieve(query, k=3):
    """Return top-k articles most similar to the query."""
    q_vec = encoder.encode([query], normalize_embeddings=True).astype(np.float32)
    scores, indices = index.search(q_vec, k)
    hits = []
    for score, idx in zip(scores[0], indices[0]):
        hits.append({
            'text':     article_texts[idx],
            'category': df['category'].iloc[idx],
            'score':    float(score),
        })
    return hits

# Quick test
hits = retrieve("Who won the football championship?", k=3)
for i, h in enumerate(hits):
    print(f"[{i+1}] ({h['category']}, score={h['score']:.3f})")
    print(f"     {h['text'][:200]}\n")

In [ ]:
# ── Generator: Flan-T5-base ───────────────────────────────────────────────────
# Running on CPU for generation (inference only — MPS T5 can be unstable)
from transformers import pipeline as hf_pipeline

print("Loading Flan-T5-base (this downloads ~990 MB on first run)...")
generator = hf_pipeline(
    'text2text-generation',
    model='google/flan-t5-base',
    device=-1,           # CPU for generation stability
    max_new_tokens=150,
)
print("Generator loaded.")

In [ ]:
# ── Prompt template ───────────────────────────────────────────────────────────
def build_prompt(query, hits):
    context_blocks = []
    for i, h in enumerate(hits):
        snippet = h['text'][:400].replace('\n', ' ')
        context_blocks.append(f"Article {i+1} [{h['category']}]: {snippet}")
    context = '\n\n'.join(context_blocks)
    prompt = (
        f"You are a knowledgeable sports and news assistant. "
        f"Answer the question using only the articles provided.\n\n"
        f"{context}\n\n"
        f"Question: {query}\n"
        f"Answer:"
    )
    return prompt

# ── Full RAG function ──────────────────────────────────────────────────────────
def rag_answer(query, k=3, max_tokens=120):
    hits    = retrieve(query, k=k)
    prompt  = build_prompt(query, hits)
    output  = generator(prompt, max_new_tokens=max_tokens, do_sample=False)
    answer  = output[0]['generated_text'].strip()
    return answer, hits

# Demo
test_query = "What happened in the Olympic swimming events?"
answer, hits = rag_answer(test_query)
print(f"Query: {test_query}")
print(f"\nAnswer: {answer}")
print(f"\nRetrieved from: {[h['category'] for h in hits]}")

---
## 5. RAG Evaluation (Step 4.2 Task 2)

We evaluate the RAG system on **10 manually curated questions** drawn from
topics that appear in the BBC corpus. Expected answers are concise summaries of
what the relevant articles say. We report **ROUGE-L** (longest common subsequence
overlap) as the quantitative metric, supplemented by a qualitative discussion.

In [ ]:
# Manually curated Q&A pairs — topics grounded in the BBC corpus
qa_pairs = [
    {"question": "Who is Michael Phelps?",
     "expected": "Michael Phelps is an American swimmer who competed in the Olympics."},
    {"question": "What is the UEFA Champions League?",
     "expected": "The UEFA Champions League is a major European club football competition."},
    {"question": "What sport does Tiger Woods play?",
     "expected": "Tiger Woods is a professional golfer."},
    {"question": "What happened in the Athens Olympics?",
     "expected": "The Athens Olympics in 2004 featured competitions in swimming athletics and other sports."},
    {"question": "Who is Ronaldinho?",
     "expected": "Ronaldinho is a Brazilian footballer known for his skill and creativity."},
    {"question": "What is the Six Nations rugby tournament?",
     "expected": "The Six Nations is an annual rugby union competition between European nations."},
    {"question": "Who is Roger Federer?",
     "expected": "Roger Federer is a Swiss professional tennis player."},
    {"question": "What is cricket's Ashes series?",
     "expected": "The Ashes is a cricket test series played between England and Australia."},
    {"question": "Who won the Formula One championship?",
     "expected": "Michael Schumacher won multiple Formula One world championships."},
    {"question": "What is the Premier League?",
     "expected": "The Premier League is the top tier of English football."},
]

print(f"Evaluation set: {len(qa_pairs)} questions")

In [ ]:
# Run RAG on all questions and score with ROUGE-L
rouge = evaluate.load('rouge')

generated, expected_list = [], []
rows = []

for qa in qa_pairs:
    gen_answer, hits = rag_answer(qa['question'], k=3)
    generated.append(gen_answer)
    expected_list.append(qa['expected'])

    scores = rouge.compute(predictions=[gen_answer],
                           references=[qa['expected']])
    rows.append({
        'Question':  qa['question'],
        'Generated': gen_answer,
        'Expected':  qa['expected'],
        'ROUGE-L':   round(scores['rougeL'], 4),
    })
    print(f"Q: {qa['question'][:60]}")
    print(f"A: {gen_answer[:120]}")
    print(f"ROUGE-L: {scores['rougeL']:.4f}\n")

eval_df = pd.DataFrame(rows)

# Overall scores
overall = rouge.compute(predictions=generated, references=expected_list)
print("\n=== Overall ROUGE Scores ===")
for k, v in overall.items():
    print(f"  {k}: {v:.4f}")

In [ ]:
# Evaluation table and chart
print("\n=== Per-question ROUGE-L ===")
print(eval_df[['Question', 'ROUGE-L']].to_string(index=False))

fig, ax = plt.subplots(figsize=(11, 4))
ax.bar(range(len(eval_df)), eval_df['ROUGE-L'], color='steelblue', alpha=0.8)
ax.axhline(eval_df['ROUGE-L'].mean(), color='tomato', linestyle='--',
           label=f"Mean ROUGE-L = {eval_df['ROUGE-L'].mean():.3f}")
ax.set_xticks(range(len(eval_df)))
ax.set_xticklabels([q[:30]+'...' for q in eval_df['Question']],
                   rotation=35, ha='right', fontsize=8)
ax.set_ylabel('ROUGE-L Score')
ax.set_title('RAG Evaluation — ROUGE-L per Question')
ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(FIGS / '18_rag_rouge_scores.png', bbox_inches='tight')
plt.show()

---
## 6. Streamlit App — BBC News Assistant (Step 4.2 Task 3)

The app is written to `app/app.py`. It provides two tabs:
- **Article Classifier** — paste any text; the app classifies it into one of 5 categories using the best ML pipeline
- **Sports Q&A** — ask a question; the RAG system retrieves relevant BBC articles and generates an answer

### Deployment to HuggingFace Spaces
1. Go to [huggingface.co/spaces](https://huggingface.co/spaces) → New Space → SDK: Streamlit
2. Upload `app/app.py` and `app/requirements.txt`
3. Upload `data/processed/faiss_index.index`, `data/processed/articles_for_rag.csv`, `data/processed/best_ml_pipeline.pkl`
4. The app auto-deploys in ~2 min

In [ ]:
app_code = '''
import streamlit as st
import pandas as pd
import numpy as np
import pickle, faiss
from pathlib import Path
from sentence_transformers import SentenceTransformer
from transformers import pipeline as hf_pipeline

# ── Page config ───────────────────────────────────────────────────────────────
st.set_page_config(page_title="BBC News Assistant", page_icon="📰", layout="wide")

# ── Load artefacts (cached) ───────────────────────────────────────────────────
@st.cache_resource
def load_classifier():
    with open("best_ml_pipeline.pkl", "rb") as f:
        data = pickle.load(f)
    return data["pipeline"], data["label_encoder"]

@st.cache_resource
def load_rag():
    encoder  = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
    index    = faiss.read_index("faiss_index.index")
    articles = pd.read_csv("articles_for_rag.csv")
    gen      = hf_pipeline("text2text-generation", model="google/flan-t5-base",
                            device=-1, max_new_tokens=150)
    return encoder, index, articles, gen

# ── Title ─────────────────────────────────────────────────────────────────────
st.title("📰 BBC News Assistant")
st.markdown("**ITC6110 NLP Group Project** — Text classification + RAG-powered Q&A")

tab1, tab2 = st.tabs(["🏷️ Article Classifier", "💬 Sports Q&A (RAG)"])

# ── Tab 1: Classifier ─────────────────────────────────────────────────────────
with tab1:
    st.subheader("Article Category Classifier")
    st.markdown("Paste any news article and the model will classify it into one of 5 BBC categories.")

    clf_pipeline, le = load_classifier()
    user_text = st.text_area("Article text", height=200,
                              placeholder="Paste a news article here...")
    if st.button("Classify", type="primary"):
        if user_text.strip():
            pred_id = clf_pipeline.predict([user_text])[0]
            pred_label = le.classes_[pred_id]
            proba = clf_pipeline.predict_proba([user_text])[0]

            col1, col2 = st.columns(2)
            col1.metric("Predicted Category", pred_label.upper())
            col2.metric("Confidence", f"{proba.max()*100:.1f}%")

            prob_df = pd.DataFrame({
                "Category": le.classes_,
                "Probability": proba
            }).sort_values("Probability", ascending=True)
            st.bar_chart(prob_df.set_index("Category"))
        else:
            st.warning("Please enter some text.")

# ── Tab 2: RAG Q&A ────────────────────────────────────────────────────────────
with tab2:
    st.subheader("Sports & News Q&A — Powered by RAG")
    st.markdown("Ask any question about sports or news. The system retrieves relevant BBC articles and generates an answer.")

    encoder, faiss_index, articles, generator = load_rag()

    query = st.text_input("Your question", placeholder="e.g. Who is Roger Federer?")
    k_val = st.slider("Articles to retrieve", min_value=1, max_value=5, value=3)

    if st.button("Ask", type="primary"):
        if query.strip():
            with st.spinner("Retrieving articles and generating answer..."):
                q_vec = encoder.encode([query], normalize_embeddings=True).astype(np.float32)
                scores, indices = faiss_index.search(q_vec, k_val)

                hits = [{"text": articles["text"].iloc[i],
                         "category": articles["category"].iloc[i],
                         "score": float(s)}
                        for s, i in zip(scores[0], indices[0])]

                context = "\\n\\n".join(
                    f"Article {i+1} [{h['category']}]: {h['text'][:400]}"
                    for i, h in enumerate(hits)
                )
                prompt = (f"You are a knowledgeable news assistant. "
                          f"Answer using only the articles provided.\\n\\n"
                          f"{context}\\n\\nQuestion: {query}\\nAnswer:")

                answer = generator(prompt, max_new_tokens=120, do_sample=False)[0]["generated_text"]

            st.success(f"**Answer:** {answer}")

            with st.expander("📄 Retrieved Articles"):
                for i, h in enumerate(hits):
                    st.markdown(f"**Article {i+1}** — *{h['category']}* (score: {h['score']:.3f})")
                    st.write(h["text"][:500] + "...")
                    st.divider()
        else:
            st.warning("Please enter a question.")
'''

app_path = Path('../app/app.py')
app_path.write_text(app_code)
print(f"Written: app/app.py  ({app_path.stat().st_size//1024} KB)")

In [ ]:
# requirements.txt for HuggingFace Spaces
req = """streamlit>=1.35
transformers>=4.40
sentence-transformers>=2.7
faiss-cpu>=1.8
torch>=2.0
datasets>=2.19
evaluate>=0.4
rouge_score>=0.1
pandas>=2.0
numpy>=1.26
"""
Path('../app/requirements.txt').write_text(req)
print("Written: app/requirements.txt")
print("\n=== Deployment checklist ===")
print("[x] app/app.py")
print("[x] app/requirements.txt")
print("[x] data/processed/faiss_index.index")
print("[x] data/processed/articles_for_rag.csv")
print("[x] data/processed/best_ml_pipeline.pkl")
print("\n→ Upload these 5 files to a new HuggingFace Space (Streamlit SDK)")

---
## 7. Summary & Artefacts

In [ ]:
print("=== Notebook 3 Complete ===\n")

print("Models trained:")
print(f"  BiLSTM          — acc={lstm_acc:.4f}  macro_f1={lstm_f1:.4f}")
print(f"  DistilBERT      — acc={bert_acc:.4f}  macro_f1={bert_f1:.4f}")

print("\nFigures generated:")
for p in sorted(FIGS.glob('1[4-9]*.png')) + sorted(FIGS.glob('17*.png')):
    print(f"  {p.name}")

print("\nSaved artefacts:")
for f in ['faiss_index.index', 'articles_for_rag.csv']:
    p = DATA / f
    print(f"  {f}  ({p.stat().st_size/1024:.0f} KB)")
for f in ['lstm_classifier.pt']:
    p = MODELS / f
    print(f"  {f}  ({p.stat().st_size/1024:.0f} KB)")
print("  models/distilbert/  (HuggingFace checkpoint)")
print("  app/app.py + app/requirements.txt")